# Notebook 16 — CGCS-Constrained Adaptive Control

This notebook turns Notebook 15's failure mode into a control rule:

> adaptive gating only helps when parameter updates remain bounded by CGCS phase-lock geometry.

Notebook 15 showed that naive adaptive gates can become too permissive after bursts and model mismatch. Notebook 16 adds a CGCS constraint around adaptive gate updates so innovation gating cannot expand without preserving phase-lock margin.

Outputs use the same repo convention:

- figures: `figures/cgcs_constrained_adaptive_control/16_cgcs_constrained_adaptive_control_*.png`
- results: `results/cgcs_constrained_adaptive_control_summary.md`
- docs: `docs/16_cgcs_constrained_adaptive_control.md`


In [ ]:
import os
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(9423)

NOTEBOOK_ID = "16"
SLUG = "cgcs_constrained_adaptive_control"
FIG_DIR = f"figures/{SLUG}"
RESULTS_DIR = "results"
DOCS_DIR = "docs"

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(DOCS_DIR, exist_ok=True)

def save_fig(name):
    # Save figure using Notebook 16 naming convention.
    png = f"{FIG_DIR}/{NOTEBOOK_ID}_{SLUG}_{name}.png"
    pdf = f"{FIG_DIR}/{NOTEBOOK_ID}_{SLUG}_{name}.pdf"
    plt.savefig(png, dpi=300, bbox_inches="tight")
    plt.savefig(pdf, bbox_inches="tight")
    print(f"saved: {png}")


## 1. Synthetic stress scenario

Notebook 16 keeps the severe disturbance pattern from Notebooks 12–15:

- calibration drift in Ω and B
- noise-burst windows
- model-mismatch windows
- outlier blocks
- response-level scoring against an oracle target

The difference is policy logic. Adaptive gate expansion is now accepted only when the estimated response remains phase-locked:

```text
cos(θ) ≥ 1 / √(1² + 1²)
```


In [ ]:
# -----------------------------
# Simulation constants
# -----------------------------
N_BLOCKS = 80
T = np.linspace(0, 10, 400)
blocks = np.arange(N_BLOCKS)

CGCS_THRESHOLD = 1 / np.sqrt(1**2 + 1**2)
TARGET_OMEGA = 1.0
TARGET_B = 0.0

noise_burst_windows = [(28, 40), (53, 66)]
model_mismatch_windows = [(28, 40), (53, 66)]
outlier_blocks = np.array([3, 5, 11, 23, 31, 38, 45, 52, 61, 67, 72])

def in_windows(i, windows):
    return any(a <= i <= b for a, b in windows)

omega_true = (
    0.23*np.sin(0.25*blocks + 0.3)
    + 0.12*np.sin(0.71*blocks - 0.7)
    + 0.008*blocks/N_BLOCKS
)
b_true = (
    0.015*np.sin(0.19*blocks + 0.4)
    + 0.00075*blocks
    + 0.006*np.sin(0.63*blocks)
)

omega_meas = omega_true + np.random.normal(0, 0.012, N_BLOCKS)
b_meas = b_true + np.random.normal(0, 0.004, N_BLOCKS)

for i in range(N_BLOCKS):
    if in_windows(i, noise_burst_windows):
        omega_meas[i] += np.random.normal(0, 0.16)
        b_meas[i] += np.random.normal(0, 0.045)
    if i in outlier_blocks:
        omega_meas[i] += np.random.choice([-1, 1]) * np.random.uniform(0.45, 0.85)
        b_meas[i] += np.random.choice([-1, 1]) * np.random.uniform(0.10, 0.22)

phase_bias = np.zeros(N_BLOCKS)
for i in range(N_BLOCKS):
    if in_windows(i, model_mismatch_windows):
        phase_bias[i] = 0.16*np.sin(0.9*i) + 0.08

print("CGCS threshold:", CGCS_THRESHOLD)
print("noise windows:", noise_burst_windows)
print("model mismatch windows:", model_mismatch_windows)
print("outlier blocks:", outlier_blocks.tolist())


## 2. Response model and policy utilities

Each policy generates a correction command. Response quality is measured by RMSE against an oracle target response and by cosine similarity against the target response.


In [ ]:
def target_response(t):
    return np.sin(TARGET_OMEGA * np.pi * t)**2

TARGET_Y = target_response(T)
TARGET_CENTERED = TARGET_Y - TARGET_Y.mean()
TARGET_NORM = np.linalg.norm(TARGET_CENTERED) + 1e-12

def response_from_params(omega_error, b_error, i):
    omega_eff = TARGET_OMEGA + omega_error + phase_bias[i]
    b_eff = TARGET_B + b_error
    y = np.sin(omega_eff * np.pi * T)**2 + b_eff
    return np.clip(y, 0, 1)

def rmse(y):
    return float(np.sqrt(np.mean((y - TARGET_Y)**2)))

def cosine_similarity(y):
    yc = y - y.mean()
    return float(np.dot(yc, TARGET_CENTERED) / ((np.linalg.norm(yc) + 1e-12) * TARGET_NORM))

def command_response(omega_cmd, b_cmd, i):
    y = response_from_params(omega_true[i] - omega_cmd, b_true[i] - b_cmd, i)
    return y

def moving_average(x, window=5):
    out = np.zeros_like(x)
    for i in range(len(x)):
        lo = max(0, i-window+1)
        out[i] = np.mean(x[lo:i+1])
    return out


## 3. Filters and controllers

Policies:

- `none`: no compensation
- `moving_average`: lagging baseline
- `robust_gated_kalman`: fixed innovation gate
- `naive_adaptive_gate_kalman`: expands/reduces gate using innovations only
- `cgcs_constrained_adaptive_gate_kalman`: accepts adaptive gate updates only if CGCS margin remains valid
- `cgcs_constrained_adaptive_joint_kalman`: same idea with joint Ω/B update
- `constrained_mpc`: conservative command smoother from prior notebooks
- `oracle`: perfect drift cancellation


In [ ]:
def robust_gated_filter(z, q=7e-4, r=2e-4, gate=0.075, init=0.0):
    x = init
    p = 1.0
    xs, rejects, gates = [], [], []
    for zi in z:
        p = p + q
        innov = zi - x
        if abs(innov) <= gate:
            k = p / (p + r)
            x = x + k*innov
            p = (1-k)*p
            rejects.append(False)
        else:
            p *= 1.02
            rejects.append(True)
        xs.append(x)
        gates.append(gate)
    return np.array(xs), np.array(rejects), np.array(gates)

def naive_adaptive_gate_filter(z, q=7e-4, r=2e-4, gate0=0.04, min_gate=0.035, max_gate=0.22, init=0.0):
    x = init
    p = 1.0
    gate = gate0
    xs, rejects, gates = [], [], []
    for zi in z:
        p = p + q
        innov = zi - x
        if abs(innov) <= gate:
            k = p / (p + r)
            x = x + k*innov
            p = (1-k)*p
            gate = min(max_gate, 0.92*gate + 0.08*max(abs(innov)*2.5, min_gate))
            rejects.append(False)
        else:
            gate = min(max_gate, gate*1.35)
            p *= 1.04
            rejects.append(True)
        xs.append(x)
        gates.append(gate)
    return np.array(xs), np.array(rejects), np.array(gates)

def cgcs_constrained_adaptive_filter(z_omega, z_b=None, q=7e-4, r=2e-4, gate0=0.04,
                                     min_gate=0.030, max_gate=0.18, margin_floor=0.025,
                                     joint=False):
    x_o = 0.0
    x_b = 0.0
    p_o = 1.0
    p_b = 1.0
    gate_o = gate0
    xs_o, xs_b, rejects, gates_o, margins = [], [], [], [], []

    if z_b is None:
        z_b = np.zeros_like(z_omega)

    for i, (zo, zb) in enumerate(zip(z_omega, z_b)):
        p_o += q
        p_b += q*0.15
        innov_o = zo - x_o
        innov_b = zb - x_b

        candidate_gate_o = gate_o
        candidate_x_o = x_o
        candidate_x_b = x_b

        if abs(innov_o) <= gate_o:
            ko = p_o / (p_o + r)
            candidate_x_o = x_o + ko*innov_o
            if joint:
                kb = p_b / (p_b + r*0.35)
                candidate_x_b = x_b + kb*innov_b
            candidate_gate_o = min(max_gate, 0.95*gate_o + 0.05*max(abs(innov_o)*2.0, min_gate))
        else:
            candidate_gate_o = min(max_gate, gate_o*1.20)

        y_candidate = command_response(candidate_x_o, candidate_x_b, i)
        candidate_margin = cosine_similarity(y_candidate) - CGCS_THRESHOLD

        if candidate_margin >= margin_floor:
            if abs(innov_o) <= gate_o:
                p_o = (1 - p_o/(p_o + r))*p_o
                if joint:
                    p_b = (1 - p_b/(p_b + r*0.35))*p_b
            x_o, x_b = candidate_x_o, candidate_x_b
            gate_o = candidate_gate_o
            accepted = True
        else:
            gate_o = max(min_gate, gate_o*0.72)
            p_o *= 1.01
            p_b *= 1.01
            accepted = False

        xs_o.append(x_o)
        xs_b.append(x_b)
        rejects.append(not accepted)
        gates_o.append(gate_o)
        margins.append(cosine_similarity(command_response(x_o, x_b, i)) - CGCS_THRESHOLD)

    return {
        "omega": np.array(xs_o),
        "b": np.array(xs_b),
        "rejects": np.array(rejects),
        "gates": np.array(gates_o),
        "margins": np.array(margins),
    }

def constrained_mpc_like(omega_est, b_est, max_delta_o=0.075, max_delta_b=0.025, smoothing=0.55):
    cmd_o = np.zeros_like(omega_est)
    cmd_b = np.zeros_like(b_est)
    for i in range(len(omega_est)):
        if i == 0:
            cmd_o[i] = np.clip(omega_est[i], -max_delta_o, max_delta_o)
            cmd_b[i] = np.clip(b_est[i], -max_delta_b, max_delta_b)
        else:
            proposed_o = smoothing*cmd_o[i-1] + (1-smoothing)*omega_est[i]
            proposed_b = smoothing*cmd_b[i-1] + (1-smoothing)*b_est[i]
            cmd_o[i] = cmd_o[i-1] + np.clip(proposed_o - cmd_o[i-1], -max_delta_o, max_delta_o)
            cmd_b[i] = cmd_b[i-1] + np.clip(proposed_b - cmd_b[i-1], -max_delta_b, max_delta_b)
    return cmd_o, cmd_b


## 4. Run policy stack

In [ ]:
ma_o = moving_average(omega_meas, window=6)
ma_b = moving_average(b_meas, window=6)

robust_o, robust_rej, robust_gate = robust_gated_filter(omega_meas, gate=0.075)
robust_b, _, _ = robust_gated_filter(b_meas, gate=0.030, q=1e-4, r=5e-5)

naive_o, naive_rej, naive_gate = naive_adaptive_gate_filter(omega_meas)
naive_b, _, _ = naive_adaptive_gate_filter(b_meas, q=1e-4, r=5e-5, gate0=0.020, min_gate=0.012, max_gate=0.09)

cgcs_scalar = cgcs_constrained_adaptive_filter(omega_meas, None, joint=False, margin_floor=0.025)
cgcs_joint = cgcs_constrained_adaptive_filter(omega_meas, b_meas, joint=True, margin_floor=0.025)

mpc_o, mpc_b = constrained_mpc_like(robust_o, robust_b, max_delta_o=0.065, max_delta_b=0.018, smoothing=0.60)

policies = {
    "none": (np.zeros(N_BLOCKS), np.zeros(N_BLOCKS)),
    "moving_average": (ma_o, ma_b),
    "robust_gated_kalman": (robust_o, robust_b),
    "naive_adaptive_gate_kalman": (naive_o, naive_b),
    "cgcs_constrained_adaptive_gate_kalman": (cgcs_scalar["omega"], np.zeros(N_BLOCKS)),
    "cgcs_constrained_adaptive_joint_kalman": (cgcs_joint["omega"], cgcs_joint["b"]),
    "constrained_mpc": (mpc_o, mpc_b),
    "oracle": (omega_true.copy(), b_true.copy()),
}

rows = []
series = {}
for name, (co, cb) in policies.items():
    ys = np.array([command_response(co[i], cb[i], i) for i in range(N_BLOCKS)])
    rmses = np.array([rmse(y) for y in ys])
    coses = np.array([cosine_similarity(y) for y in ys])
    margins = coses - CGCS_THRESHOLD
    series[name] = {"y": ys, "rmse": rmses, "cos": coses, "margin": margins, "omega_cmd": co, "b_cmd": cb}
    rows.append({
        "policy": name,
        "mean_rmse": float(np.mean(rmses)),
        "max_rmse": float(np.max(rmses)),
        "min_cosine": float(np.min(coses)),
        "mean_cosine": float(np.mean(coses)),
        "blocks_below_threshold": int(np.sum(coses < CGCS_THRESHOLD)),
    })

summary_df = pd.DataFrame(rows).sort_values("mean_rmse")
summary_df


## 5. Figures

In [ ]:
def shade_windows(alpha1=0.18, alpha2=0.10):
    ax = plt.gca()
    for a, b in noise_burst_windows:
        ax.axvspan(a, b, alpha=alpha1, label="noise burst" if a == noise_burst_windows[0][0] else None)
    for a, b in model_mismatch_windows:
        ax.axvspan(a, b, alpha=alpha2, label="model mismatch" if a == model_mismatch_windows[0][0] else None)

plot_order = ["none", "moving_average", "robust_gated_kalman", "naive_adaptive_gate_kalman", "cgcs_constrained_adaptive_gate_kalman", "cgcs_constrained_adaptive_joint_kalman", "constrained_mpc", "oracle"]

plt.figure(figsize=(12, 5))
plt.plot(blocks, omega_true, label="true Ω drift", linewidth=2)
plt.plot(blocks, omega_meas, "o-", label="measured Ω", alpha=0.75)
plt.plot(blocks, robust_o, label="robust gated Kalman")
plt.plot(blocks, naive_o, label="naive adaptive gate Kalman")
plt.plot(blocks, cgcs_scalar["omega"], label="CGCS constrained adaptive gate")
plt.plot(blocks, cgcs_joint["omega"], label="CGCS constrained adaptive joint")
plt.plot(blocks, mpc_o, label="constrained MPC")
shade_windows()
plt.axhline(0, linewidth=1)
plt.title("CGCS-constrained adaptive control: Ω tracking")
plt.xlabel("calibration block")
plt.ylabel("Ω drift estimate / command")
plt.legend(ncol=2)
save_fig("omega_tracking")
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(blocks, b_true, label="true B drift", linewidth=2)
plt.plot(blocks, b_meas, "o-", label="measured B", alpha=0.65)
plt.plot(blocks, robust_b, label="robust gated Kalman")
plt.plot(blocks, cgcs_joint["b"], label="CGCS constrained adaptive joint")
plt.plot(blocks, mpc_b, label="constrained MPC")
shade_windows()
plt.axhline(0, linewidth=1)
plt.title("CGCS-constrained adaptive control: B tracking")
plt.xlabel("calibration block")
plt.ylabel("B drift estimate / command")
plt.legend()
save_fig("offset_tracking")
plt.show()

plt.figure(figsize=(12, 5))
for name in plot_order:
    plt.plot(blocks, series[name]["rmse"], marker="o", label=name)
shade_windows()
plt.title("CGCS-constrained adaptive control: response RMSE comparison")
plt.xlabel("calibration block")
plt.ylabel("RMSE vs target response")
plt.legend(ncol=2)
save_fig("response_rmse_comparison")
plt.show()

plt.figure(figsize=(12, 5))
for name in plot_order:
    plt.plot(blocks, series[name]["cos"], marker="o", label=name)
plt.axhline(CGCS_THRESHOLD, linestyle="--", label="45° threshold")
shade_windows()
plt.title("CGCS-constrained adaptive control: CGCS phase-lock stability")
plt.xlabel("calibration block")
plt.ylabel("cosine similarity vs target")
plt.legend(ncol=2)
save_fig("cgcs_stability_comparison")
plt.show()

plt.figure(figsize=(12, 5))
for name in plot_order[:-1]:
    plt.plot(blocks, series[name]["margin"], marker="o", label=name)
plt.axhline(0, linestyle="--", label="threshold margin = 0")
shade_windows()
plt.title("CGCS-constrained adaptive control: threshold margin")
plt.xlabel("calibration block")
plt.ylabel("CGCS margin above threshold")
plt.legend(ncol=2)
save_fig("threshold_margin")
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(blocks, robust_gate, "--", label="fixed robust Ω gate")
plt.plot(blocks, naive_gate, label="naive adaptive Ω gate")
plt.plot(blocks, cgcs_scalar["gates"], label="CGCS constrained adaptive Ω gate")
plt.plot(blocks, cgcs_joint["gates"], label="CGCS constrained adaptive joint Ω gate")
shade_windows()
plt.title("CGCS-constrained adaptive control: gate trace")
plt.xlabel("calibration block")
plt.ylabel("Ω innovation gate")
plt.legend()
save_fig("gate_trace")
plt.show()

plt.figure(figsize=(12, 4))
rej_series = {
    "robust_gated_kalman": robust_rej,
    "naive_adaptive_gate_kalman": naive_rej,
    "cgcs_constrained_adaptive_gate_kalman": cgcs_scalar["rejects"],
    "cgcs_constrained_adaptive_joint_kalman": cgcs_joint["rejects"],
}
for j, (name, rej) in enumerate(rej_series.items()):
    idx = np.where(rej)[0]
    plt.scatter(idx, np.full_like(idx, j), marker="x", s=70, label=name)
plt.yticks(range(len(rej_series)), list(rej_series.keys()))
shade_windows(alpha1=0.08, alpha2=0.05)
plt.title("CGCS-constrained adaptive control: rejection events")
plt.xlabel("calibration block")
plt.ylabel("policy")
plt.legend(loc="upper right")
save_fig("rejection_events")
plt.show()

rank = summary_df.copy()
plt.figure(figsize=(12, 5))
x = np.arange(len(rank)); w = 0.38
plt.bar(x - w/2, rank["mean_rmse"], width=w, label="mean RMSE")
plt.bar(x + w/2, rank["max_rmse"], width=w, label="max RMSE")
plt.xticks(x, rank["policy"], rotation=25, ha="right")
plt.ylabel("response RMSE")
plt.title("CGCS-constrained adaptive control: policy ranking")
plt.legend()
save_fig("policy_ranking_summary")
plt.show()

plt.figure(figsize=(12, 4.8))
fail_matrix = np.array([series[name]["cos"] < CGCS_THRESHOLD for name in plot_order])
plt.imshow(fail_matrix, aspect="auto", interpolation="nearest")
plt.yticks(np.arange(len(plot_order)), plot_order)
plt.xlabel("calibration block")
plt.ylabel("policy")
plt.title("CGCS-constrained adaptive control: failure-event map")
cbar = plt.colorbar(); cbar.set_label("failure: cosθ < threshold")
save_fig("failure_event_map")
plt.show()

worst_policy = summary_df[summary_df["policy"] != "oracle"].sort_values("max_rmse", ascending=False).iloc[0]["policy"]
worst_block = int(np.argmax(series[worst_policy]["rmse"]))
plt.figure(figsize=(12, 5))
plt.plot(T, TARGET_Y, label="target", linewidth=2)
for name in plot_order:
    plt.plot(T, series[name]["y"][worst_block], label=name, alpha=0.9)
plt.title(f"CGCS-constrained adaptive control: worst-case block {worst_block}")
plt.xlabel("pulse duration")
plt.ylabel("excited-state probability")
plt.legend(ncol=2)
save_fig("worst_case_block_comparison")
plt.show()


## 6. Save results and markdown

In [ ]:
summary_path = f"{RESULTS_DIR}/{SLUG}_summary.csv"
summary_df.to_csv(summary_path, index=False)
print(f"saved: {summary_path}")

json_path = f"{RESULTS_DIR}/{SLUG}_metrics.json"
with open(json_path, "w") as f:
    json.dump({
        "notebook": NOTEBOOK_ID,
        "slug": SLUG,
        "cgcs_threshold": CGCS_THRESHOLD,
        "noise_burst_windows": noise_burst_windows,
        "model_mismatch_windows": model_mismatch_windows,
        "outlier_blocks": outlier_blocks.tolist(),
        "summary": summary_df.to_dict(orient="records"),
    }, f, indent=2)
print(f"saved: {json_path}")

figures = [
    ("Omega tracking", "omega_tracking", "CGCS-constrained adaptive policies track Ω while blocking unsafe gate expansion."),
    ("Offset tracking", "offset_tracking", "Joint CGCS-constrained adaptation tracks B without letting offset noise dominate response control."),
    ("Response RMSE comparison", "response_rmse_comparison", "Response-level error shows where naive adaptation creates unstable error spikes."),
    ("CGCS phase-lock stability", "cgcs_stability_comparison", "CGCS-constrained filters preserve margin more consistently than naive adaptive gating."),
    ("Threshold margin", "threshold_margin", "Positive margin means policy remains above the 45° phase-lock threshold."),
    ("Gate trace", "gate_trace", "Gate traces show how CGCS constraints prevent unchecked gate expansion."),
    ("Rejection events", "rejection_events", "Rejected updates cluster around noisy or mismatched regions."),
    ("Policy ranking", "policy_ranking_summary", "Ranking compares mean and worst-case response RMSE."),
    ("Failure-event map", "failure_event_map", "Failure map marks blocks where cosine similarity falls below the CGCS threshold."),
    ("Worst-case block", "worst_case_block_comparison", "Worst-case response comparison shows phase-shifted or distorted pulse behavior."),
]

figure_md = "\n\n".join(
    f"### {title}\n\n![{title}](../figures/{SLUG}/{NOTEBOOK_ID}_{SLUG}_{name}.png)\n\n{caption}"
    for title, name, caption in figures
)

ranking_md = summary_df.to_markdown(index=False)

summary_md = f"""# CGCS-Constrained Adaptive Control Results Summary

## Configuration

- Notebook: {NOTEBOOK_ID}
- Slug: `{SLUG}`
- Blocks: {N_BLOCKS}
- CGCS threshold: {CGCS_THRESHOLD:.4f}
- Noise burst windows: {noise_burst_windows}
- Model mismatch windows: {model_mismatch_windows}
- Outlier blocks: {outlier_blocks.tolist()}

---

## Policy Ranking

{ranking_md}

---

## Key Observations

### Notebook 15 failure mode

Naive adaptive gates can expand after repeated innovation misses. That helps in calm regions, but under burst noise and model mismatch it admits unsafe measurements and can reduce phase-lock margin.

### Notebook 16 correction

CGCS-constrained adaptation tests candidate updates against the phase-lock threshold before accepting gate expansion or measurement updates.

### Control interpretation

Estimator adaptation is useful only when it is geometrically bounded. The gate should not merely follow innovation size; it should preserve response-level phase alignment.

---

## Next Direction

Notebook 17 can package the controller as a reusable API:

- `fit()` measurement calibration
- `predict()` drift forecast
- `command()` bounded compensation
- `score()` CGCS, RMSE, and recovery diagnostics
"""

result_md_path = f"{RESULTS_DIR}/{SLUG}_summary.md"
with open(result_md_path, "w") as f:
    f.write(summary_md)
print(f"saved: {result_md_path}")

doc_md = f"""# CGCS-Constrained Adaptive Control (Notebook 16)

Notebook 16 follows Notebook 15 by turning adaptive-gate failure into a constraint rule.

Adaptive gates are useful only when parameter updates remain bounded by CGCS phase-lock geometry. The notebook compares naive adaptive gating with CGCS-constrained adaptive Kalman policies.

---

## Model

Policies estimate Ω drift and B offset drift, then generate compensation commands. Response quality is evaluated against an oracle target response.

A candidate adaptive update is accepted only if:

```text
cos(θ) ≥ 1 / √(1² + 1²)
```

In practical terms: gate expansion must preserve phase-lock margin.

---

## Figures

{figure_md}

---

## Results

{ranking_md}

---

## Key Takeaways

- Naive adaptive gating can become too permissive after burst and mismatch regimes.
- CGCS-constrained adaptation blocks unsafe gate expansion.
- Robust gating remains a strong baseline.
- Joint adaptive filtering is useful only when its update rule is constrained by response-level geometry.
- Control-stack performance depends on estimator structure plus acceptance rules, not estimator smoothness alone.

---

## Next Step

Notebook 17 should convert the best control logic into a reusable controller interface for later hardware and QCVV-style experiments.
"""

doc_md_path = f"{DOCS_DIR}/{NOTEBOOK_ID}_{SLUG}.md"
with open(doc_md_path, "w") as f:
    f.write(doc_md)
print(f"saved: {doc_md_path}")


## 7. Optional Colab download zip

Run this cell in Colab after all figures/results/docs are generated.


In [ ]:
# Optional in Colab:
# !zip -r 16_cgcs_constrained_adaptive_control_outputs.zip #   figures/cgcs_constrained_adaptive_control #   results/cgcs_constrained_adaptive_control_summary.csv #   results/cgcs_constrained_adaptive_control_metrics.json #   results/cgcs_constrained_adaptive_control_summary.md #   docs/16_cgcs_constrained_adaptive_control.md
#
# from google.colab import files
# files.download('16_cgcs_constrained_adaptive_control_outputs.zip')
